# Hummingbot Trade Profitability Analysis

This notebook loads `TradeFill` records from a Hummingbot SQLite DB and computes:
- Net quote PnL from fills (including fees when available)
- Inventory carry (net base position)
- Mark-to-market PnL at the latest fill price
- Hourly and daily profitability breakdown


In [ ]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


In [ ]:
# ---- Parameters ----
DB_PATH = ''  # e.g. '/home/sophos9/quanta_mm/data/quanta_mm.sqlite'
EXCHANGE = 'cryptocom'
SYMBOL = 'BTC-USD'
STRATEGY = None      # e.g. 'avellaneda_market_making'
CONFIG_FILE = None   # e.g. 'quanta_mm.yml'
START = None         # e.g. '2026-02-15 00:00:00'
END = None           # e.g. '2026-02-16 00:00:00'

# Optional: if you want 'vs hold' from known session start balances
START_BASE_BALANCE = None
START_QUOTE_BALANCE = None


In [ ]:
def find_db_candidates():
    roots = [Path('.'), Path.home() / 'hummingbot_files', Path.home() / 'data', Path('/home')]
    cands = []
    for r in roots:
        if not r.exists():
            continue
        try:
            cands.extend(list(r.rglob('*.sqlite')))
        except Exception:
            pass
    return sorted({str(p) for p in cands})

if not DB_PATH:
    candidates = find_db_candidates()
    print('DB_PATH is empty. Candidates found:')
    for c in candidates[:30]:
        print(' -', c)
    if candidates:
        DB_PATH = candidates[0]
        print('Using first candidate:', DB_PATH)

assert DB_PATH and Path(DB_PATH).exists(), f'Database not found: {DB_PATH}'


In [ ]:
def load_trade_fills(db_path, exchange=None, symbol=None, strategy=None, config_file=None, start=None, end=None):
    conn = sqlite3.connect(db_path)
    query = """
    SELECT
      config_file_path, strategy, market, symbol, base_asset, quote_asset,
      timestamp, order_id, trade_type, order_type, price, amount,
      trade_fee, trade_fee_in_quote, exchange_trade_id, position
    FROM TradeFill
    WHERE 1=1
    """
    params = []
    if exchange:
        query += " AND market = ?"
        params.append(exchange)
    if symbol:
        query += " AND symbol = ?"
        params.append(symbol)
    if strategy:
        query += " AND strategy = ?"
        params.append(strategy)
    if config_file:
        query += " AND config_file_path = ?"
        params.append(config_file)
    if start:
        query += " AND timestamp >= ?"
        params.append(int(pd.Timestamp(start).timestamp() * 1000))
    if end:
        query += " AND timestamp <= ?"
        params.append(int(pd.Timestamp(end).timestamp() * 1000))
    query += " ORDER BY timestamp ASC"

    df = pd.read_sql_query(query, conn, params=params)
    conn.close()
    if df.empty:
        return df

    df['timestamp_ms'] = df['timestamp'].astype('int64')
    df['dt'] = pd.to_datetime(df['timestamp_ms'], unit='ms', utc=True)
    df['price'] = df['price'].astype(float)
    df['amount'] = df['amount'].astype(float)
    df['side'] = df['trade_type'].str.lower()
    df['is_buy'] = df['side'].eq('buy')
    df['signed_base'] = np.where(df['is_buy'], df['amount'], -df['amount'])
    df['quote_notional'] = df['price'] * df['amount']
    # Positive quote flow means cash in (sell), negative means cash out (buy)
    df['signed_quote_flow_gross'] = np.where(df['is_buy'], -df['quote_notional'], df['quote_notional'])
    return df


In [ ]:
def parse_fee_in_quote(row):
    # 1) Prefer precomputed trade_fee_in_quote
    if row.get('trade_fee_in_quote') is not None and str(row.get('trade_fee_in_quote')) != '':
        try:
            return float(row['trade_fee_in_quote'])
        except Exception:
            pass

    # 2) Try parsing JSON trade_fee blob
    raw = row.get('trade_fee')
    if raw is None or raw == '':
        return 0.0

    try:
        fee_obj = raw if isinstance(raw, dict) else json.loads(raw)
    except Exception:
        return 0.0

    quote_asset = str(row.get('quote_asset', '')).upper()
    base_asset = str(row.get('base_asset', '')).upper()
    price = float(row['price'])
    amount = float(row['amount'])
    notional = price * amount

    total_fee_quote = 0.0

    # Percent fee
    pct = fee_obj.get('percent') if isinstance(fee_obj, dict) else None
    try:
        if pct is not None:
            total_fee_quote += float(pct) * notional
    except Exception:
        pass

    # Flat fees
    flats = fee_obj.get('flat_fees', []) if isinstance(fee_obj, dict) else []
    for f in flats:
        try:
            token = str(f.get('token', '')).upper()
            amt = float(f.get('amount', 0))
            if token == quote_asset:
                total_fee_quote += amt
            elif token == base_asset:
                total_fee_quote += amt * price
        except Exception:
            continue

    return total_fee_quote


In [ ]:
df = load_trade_fills(DB_PATH, EXCHANGE, SYMBOL, STRATEGY, CONFIG_FILE, START, END)
if df.empty:
    raise RuntimeError('No trades found for the selected filters.')

df['fee_quote'] = df.apply(parse_fee_in_quote, axis=1)
df['signed_quote_flow_net'] = df['signed_quote_flow_gross'] - df['fee_quote']

# Running positions
df['cum_base'] = df['signed_base'].cumsum()
df['cum_quote_net'] = df['signed_quote_flow_net'].cumsum()

last_price = float(df['price'].iloc[-1])
first_price = float(df['price'].iloc[0])
net_base = float(df['signed_base'].sum())
gross_quote = float(df['signed_quote_flow_gross'].sum())
fees_quote = float(df['fee_quote'].sum())
net_quote = float(df['signed_quote_flow_net'].sum())

# Fill-based PnL decomposition
inventory_mtm = net_base * last_price
total_pnl_quote = net_quote + inventory_mtm

summary = pd.DataFrame([
    {
        'db_path': DB_PATH,
        'exchange': EXCHANGE,
        'symbol': SYMBOL,
        'rows': len(df),
        'start': df['dt'].min(),
        'end': df['dt'].max(),
        'first_price': first_price,
        'last_price': last_price,
        'net_base': net_base,
        'gross_quote_flow': gross_quote,
        'fees_quote': fees_quote,
        'net_quote_flow': net_quote,
        'inventory_mtm_at_last': inventory_mtm,
        'total_pnl_quote': total_pnl_quote,
    }
])
summary.T


In [ ]:
# Optional: compare against hold if you provide session-start balances
if START_BASE_BALANCE is not None and START_QUOTE_BALANCE is not None:
    start_hold_value = float(START_BASE_BALANCE) * first_price + float(START_QUOTE_BALANCE)
    end_hold_value = float(START_BASE_BALANCE) * last_price + float(START_QUOTE_BALANCE)

    end_base_balance = float(START_BASE_BALANCE) + net_base
    end_quote_balance = float(START_QUOTE_BALANCE) + net_quote
    end_strategy_value = end_base_balance * last_price + end_quote_balance

    hold_delta = end_hold_value - start_hold_value
    strategy_delta = end_strategy_value - start_hold_value
    edge_vs_hold = end_strategy_value - end_hold_value

    pd.DataFrame([{
        'start_hold_value': start_hold_value,
        'end_hold_value': end_hold_value,
        'end_strategy_value': end_strategy_value,
        'hold_delta': hold_delta,
        'strategy_delta': strategy_delta,
        'edge_vs_hold': edge_vs_hold,
    }]).T
else:
    print('Set START_BASE_BALANCE and START_QUOTE_BALANCE to compute hold-vs-strategy.')


In [ ]:
# Time-bucket profitability
df['hour'] = df['dt'].dt.floor('h')
df['day'] = df['dt'].dt.floor('d')

hourly = df.groupby('hour', as_index=False).agg(
    trades=('exchange_trade_id', 'count'),
    gross_quote=('signed_quote_flow_gross', 'sum'),
    fees=('fee_quote', 'sum'),
    net_quote=('signed_quote_flow_net', 'sum'),
    net_base=('signed_base', 'sum'),
)
hourly['inventory_mtm_eod_hour'] = hourly['net_base'].cumsum() * last_price
hourly['cum_net_quote'] = hourly['net_quote'].cumsum()
hourly['cum_total_pnl'] = hourly['cum_net_quote'] + hourly['inventory_mtm_eod_hour']

daily = df.groupby('day', as_index=False).agg(
    trades=('exchange_trade_id', 'count'),
    gross_quote=('signed_quote_flow_gross', 'sum'),
    fees=('fee_quote', 'sum'),
    net_quote=('signed_quote_flow_net', 'sum'),
    net_base=('signed_base', 'sum'),
)

display(hourly.tail(24))
display(daily)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df['dt'], df['price'], label='Fill price')
axes[0].set_title('Fill Prices')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['dt'], df['cum_quote_net'], label='Cumulative quote flow (net fees)')
axes[1].plot(df['dt'], df['cum_base'] * last_price, label='Inventory MTM @ last price')
axes[1].plot(df['dt'], df['cum_quote_net'] + df['cum_base'] * last_price, label='Total PnL proxy')
axes[1].set_title('PnL Components')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Detailed recent trades
cols = [
    'dt', 'market', 'symbol', 'trade_type', 'price', 'amount',
    'quote_notional', 'fee_quote', 'signed_quote_flow_net', 'cum_base', 'cum_quote_net'
]
df[cols].tail(100)
